# ISLES'24 NCCT + perfusion nnU-Net five-fold CV

GPU training for `Dataset505_ISLES2024_NCCT_PERFUSION` using **NCCT + Tmax + CBF + CBV + MTT** and exactly the same patient folds as the frozen NCCT-only baseline.

To make Colab compute usage predictable, each **Run all** trains only the **next unfinished fold**. Completed folds are detected from persistent `checkpoint_final.pth` files. If a session disconnects, rerun this notebook and the same fold resumes from its latest checkpoint. Preprocessed tensors are cached on local SSD to avoid Google Drive FUSE random-access failures.

**Recommended runtime: G4 GPU, High-RAM off.**


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Persistent paths and GPU check


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess

ROOT = Path('/content/drive/MyDrive/HybridStrokeSeg/ISLES2024')
NNUNET_RAW = ROOT / 'nnUNet_raw'
NNUNET_PREPROCESSED = ROOT / 'nnUNet_preprocessed'
NNUNET_RESULTS = ROOT / 'nnUNet_results'
RESULTS = ROOT / 'results'
STATUS = RESULTS / 'nnunet_ncct_perfusion_cv_status.json'
LOG = RESULTS / 'nnunet_ncct_perfusion_cv.log'
LOCAL_PREPROCESSED_CACHE = Path('/content/nnUNet_preprocessed_cache_505')
REPO_ZIP = Path('/content/HybridStrokeSeg-main.zip')
REPO = Path('/content/HybridStrokeSeg-Reproduction-main')
DATASET = NNUNET_RAW / 'Dataset505_ISLES2024_NCCT_PERFUSION'

for path in (NNUNET_PREPROCESSED, NNUNET_RESULTS, RESULTS):
    path.mkdir(parents=True, exist_ok=True)
if not DATASET.is_dir():
    raise FileNotFoundError(
        'Dataset505 is not staged. Run notebook 13_isles2024_NCCT_PERFUSION_PREP first.'
    )

os.environ['nnUNet_raw'] = str(NNUNET_RAW)
os.environ['nnUNet_preprocessed'] = str(NNUNET_PREPROCESSED)
os.environ['nnUNet_results'] = str(NNUNET_RESULTS)

import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > G4 GPU, then Run all again.')
print('GPU:', torch.cuda.get_device_name(0))


## Load latest repository and pinned nnU-Net


In [ ]:
def run(cmd, *, cwd=None):
    cmd = [str(x) for x in cmd]
    print('$', ' '.join(cmd), flush=True)
    subprocess.run(cmd, check=True, cwd=cwd)

if REPO.exists():
    shutil.rmtree(REPO)
if REPO_ZIP.exists():
    REPO_ZIP.unlink()

run([
    'curl', '-fL', '--retry', '20', '--retry-delay', '3', '--retry-all-errors',
    'https://github.com/NTinkicht/HybridStrokeSeg-Reproduction/archive/refs/heads/main.zip',
    '-o', REPO_ZIP,
])
run(['unzip', '-q', REPO_ZIP, '-d', '/content'])
run(['python', '-m', 'pip', 'install', '-q', '-e', '.[modern]'], cwd=REPO)
run(['python', '-c', 'import nnunetv2; print("nnU-Net v2 import OK")'])


## Reconfirm split identity and select the next unfinished fold

This guard prevents an invalid ablation comparison if Dataset505 ever drifts from Dataset504 patient assignments.


In [ ]:
dataset504 = NNUNET_RAW / 'Dataset504_ISLES2024_NCCT'
run([
    'python', 'scripts/verify_nnunet_splits.py',
    dataset504 / 'splits_final.json',
    DATASET / 'splits_final.json',
], cwd=REPO)

results_dataset = NNUNET_RESULTS / DATASET.name
def fold_complete(fold):
    if not results_dataset.is_dir():
        return False
    return any(results_dataset.glob(f'*/fold_{fold}/checkpoint_final.pth'))

completed = [fold for fold in range(5) if fold_complete(fold)]
unfinished = [fold for fold in range(5) if not fold_complete(fold)]
print('Completed folds:', completed)
if not unfinished:
    print('All five Dataset505 folds are already complete. Nothing to train.')
    NEXT_FOLD = None
else:
    NEXT_FOLD = unfinished[0]
    print('This session will train/resume fold:', NEXT_FOLD)


## Train/resume one fold with persistent logging


In [ ]:
def run_logged(cmd):
    cmd = [str(x) for x in cmd]
    print('$', ' '.join(cmd), flush=True)
    with LOG.open('a', encoding='utf-8') as log_handle:
        log_handle.write('\n$ ' + ' '.join(cmd) + '\n')
        log_handle.flush()
        proc = subprocess.Popen(
            cmd,
            cwd=REPO,
            env=os.environ.copy(),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            log_handle.write(line)
            log_handle.flush()
        code = proc.wait()
        if code != 0:
            raise subprocess.CalledProcessError(code, cmd)

if NEXT_FOLD is not None:
    run_logged([
        'python', 'scripts/run_isles2024_nnunet_cv.py',
        '--dataset-id', '505',
        '--configuration', '3d_fullres',
        '--folds', str(NEXT_FOLD),
        '--status-json', STATUS,
        '--local-preprocessed-cache', LOCAL_PREPROCESSED_CACHE,
        '--cache-copy-retries', '3',
    ])


## Persistent status


In [ ]:
completed_after = [fold for fold in range(5) if fold_complete(fold)]
print('Completed folds now:', completed_after)
print('Remaining folds:', [fold for fold in range(5) if fold not in completed_after])
print('Persistent log:', LOG)
print('Persistent status:', STATUS)
if STATUS.exists():
    print(json.dumps(json.loads(STATUS.read_text()), indent=2))
if len(completed_after) < 5:
    print('\nFor the next fold, start a fresh G4 session and Run all again.')
else:
    print('\nDataset505 five-fold training is complete.')
